# FASE 3: Limpieza, Tratamiento de Nulos y Datos Faltantes

Este notebook aborda la limpieza profunda de los 3 datasets integrados con Demografía generados en la FASE 2:
- `demodietary_master`
- `demoexamination_master`
- `demolaboratory_master`

**Regla NHANES:** Los valores codificados como `7`, `77`, `777`, `7777`, `77777` (Refused) y `9`, `99`, `999`, `9999`, `99999` (Don't Know) deben tratarse como valores faltantes reales (`NaN`).

**Estrategia de limpieza por etapas:**
1. Diagnóstico inicial de nulidad
2. Recodificación de valores especiales NHANES
3. Eliminación de columnas con exceso de nulidad (umbral configurable)
4. Eliminación de filas con exceso de nulidad
5. Reporte final de calidad

In [1]:
# Inicializar Kedro
%load_ext kedro.ipython

[06/13/26 18:12:45] INFO     Using                                                                  ]8;id=671240;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=92075;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packa                
                             ges\kedro\framework\project\rich_logging.yml' as logging                              
                             configuration.                                                                        

                    INFO     Registered line magic '%reload_kedro'                                   ]8;id=767188;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=864394;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#64\64]8;;\

                    INFO     Registered line magic '%load_node'                                      ]8;id=411378;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=436650;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#66\66]8;;\

                    INFO     Resolved project path as:                                              ]8;id=311326;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=643530;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#181\181]8;;\
                             c:\Users\alarc\OneDrive\Escritorio\Proyecto_Nhanes_Kedro\proyecto_nhan                
                             es_kedro.                                                                             
                             To set a different path, run '%reload_kedro <project_root>'                           

[06/13/26 18:12:47] WARNING  c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packag ]8;id=838206;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\warnings.py\warnings.py]8;;\:]8;id=588809;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\warnings.py#112\112]8;;\
                             es\kedro\framework\project\__init__.py:350: UserWarning: The                          
                             'proyecto_nhanes_kedro.pipelines.data_science' module does not expose                 
                             a 'create_pipeline' function, so no pipelines defined therein will be                 
                             returned by 'find_pipelines'.                                                         
                               warnings.warn(                                                                      
                                                                                                                   

                    WARNING  c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packag ]8;id=417344;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\warnings.py\warnings.py]8;;\:]8;id=646285;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\warnings.py#112\112]8;;\
                             es\kedro\framework\project\__init__.py:350: UserWarning: The                          
                             'proyecto_nhanes_kedro.pipelines.feature_engineering' module does not                 
                             expose a 'create_pipeline' function, so no pipelines defined therein                  
                             will be returned by 'find_pipelines'.                                                 
                               warnings.warn(                                                                      
                                                                                                                   

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=275767;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=602183;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

[06/13/26 18:12:49] INFO     Kedro project Proyecto NHANES Kedro                                    ]8;id=150878;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=345980;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=555981;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=725372;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\ipython\__init__.py#148\148]8;;\
                             'pipelines'                                                                           

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar los 3 datasets desde el Catalog de Kedro
df_dietary = catalog.load('demodietary_master')
df_examination = catalog.load('demoexamination_master')
df_laboratory = catalog.load('demolaboratory_master')

print('--- SHAPES INICIALES ---')
print(f'DemoDietary:     {df_dietary.shape}')
print(f'DemoExamination: {df_examination.shape}')
print(f'DemoLaboratory:  {df_laboratory.shape}')

[06/13/26 18:12:51] INFO     Loading data from demodietary_master (ParquetDataset)...          ]8;id=221099;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=971604;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

                    INFO     Loading data from demoexamination_master (ParquetDataset)...      ]8;id=457868;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=189760;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

                    INFO     Loading data from demolaboratory_master (ParquetDataset)...       ]8;id=951171;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=400607;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

--- SHAPES INICIALES ---
DemoDietary:     (9254, 718)
DemoExamination: (9254, 1713)
DemoLaboratory:  (9254, 698)


## Paso 3.1: Diagnóstico Inicial de Nulidad
Revisamos el porcentaje de valores nulos por columna para cada dataset.

In [3]:
def null_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Genera un reporte de nulidad por columna."""
    total = len(df)
    null_count = df.isnull().sum()
    null_pct = (null_count / total * 100).round(2)
    report = pd.DataFrame({
        'columna': null_count.index,
        'nulos': null_count.values,
        'pct_nulos': null_pct.values,
        'dtype': df.dtypes.values
    }).sort_values('pct_nulos', ascending=False).reset_index(drop=True)
    
    total_cols = len(df.columns)
    cols_with_nulls = (null_count > 0).sum()
    cols_above_50 = (null_pct > 50).sum()
    cols_above_80 = (null_pct > 80).sum()
    
    print(f'\n=== REPORTE DE NULIDAD: {name} ===')
    print(f'Total columnas: {total_cols}')
    print(f'Columnas con al menos 1 nulo: {cols_with_nulls} ({cols_with_nulls/total_cols*100:.1f}%)')
    print(f'Columnas con >50% nulos: {cols_above_50}')
    print(f'Columnas con >80% nulos: {cols_above_80}')
    print(f'\nTop 20 columnas con mas nulos:')
    return report

report_dietary = null_report(df_dietary, 'DemoDietary')
report_dietary.head(20)


=== REPORTE DE NULIDAD: DemoDietary ===
Total columnas: 718
Columnas con al menos 1 nulo: 692 (96.4%)
Columnas con >50% nulos: 350
Columnas con >80% nulos: 283

Top 20 columnas con mas nulos:


,columna,nulos,pct_nulos,dtype
0,DSD128GG,9253,99.99,float64
1,DSD128LL,9252,99.98,float64
2,DRQSDT5,9252,99.98,float64
3,DSD128X,9252,99.98,float64
4,DSD128W,9251,99.97,float64
5,DSD128II,9250,99.96,float64
6,DS1ICAFF,9250,99.96,float64
7,DRQSDT6,9250,99.96,float64
8,DSD128PP,9249,99.95,float64
9,DS1IMFAT,9249,99.95,float64


In [4]:
report_examination = null_report(df_examination, 'DemoExamination')
report_examination.head(20)


=== REPORTE DE NULIDAD: DemoExamination ===
Total columnas: 1713
Columnas con al menos 1 nulo: 1691 (98.7%)
Columnas con >50% nulos: 1452
Columnas con >80% nulos: 296

Top 20 columnas con mas nulos:


,columna,nulos,pct_nulos,dtype
0,BMIHEAD,9254,100.00,float64
1,AUXR1K2R,9252,99.98,float64
2,AUXR1K2L,9252,99.98,float64
3,AUXR4KL,9249,99.95,float64
4,AUDROABC,9248,99.94,float64
5,AUDLOABC,9248,99.94,float64
6,AUXR3KL,9248,99.94,float64
7,AUXR8KR,9248,99.94,float64
8,AUXR6KL,9247,99.92,float64
9,AUXR3KR,9246,99.91,float64


In [5]:
report_laboratory = null_report(df_laboratory, 'DemoLaboratory')
report_laboratory.head(20)


=== REPORTE DE NULIDAD: DemoLaboratory ===
Total columnas: 698
Columnas con al menos 1 nulo: 676 (96.8%)
Columnas con >50% nulos: 444
Columnas con >80% nulos: 48

Top 20 columnas con mas nulos:


,columna,nulos,pct_nulos,dtype
0,LBXHNAT,9252,99.98,float64
1,LBXHIV1,9237,99.82,float64
2,LBXHIV2,9237,99.82,float64
3,URXVOL3,9221,99.64,float64
4,URDFLOW3,9221,99.64,float64
5,URDTIME3,9221,99.64,float64
6,PHAALCHR,9215,99.58,float64
7,PHAALCMN,9215,99.58,float64
8,LBXHCG,9205,99.47,float64
9,PHAANTMN,9189,99.30,float64


## Paso 3.2: Recodificación de Valores Especiales NHANES

Según la documentación oficial de NHANES, muchos campos numéricos codifican respuestas especiales:
- **7, 77, 777, 7777, 77777** = Refused (el encuestado se negó a responder)
- **9, 99, 999, 9999, 99999** = Don't Know (el encuestado no sabe)

Estos valores deben convertirse a `NaN` ya que NO representan datos reales.

In [6]:
# Valores especiales NHANES que representan 'Refused' o 'Don't Know'
NHANES_SPECIAL_VALUES = [7, 9, 77, 99, 777, 999, 7777, 9999, 77777, 99999]

def recode_nhanes_special_values(df: pd.DataFrame, special_values: list) -> pd.DataFrame:
    """
    Reemplaza valores especiales NHANES por NaN en columnas numéricas.
    Solo aplica a columnas numéricas para no afectar identificadores.
    Excluye SEQN y columnas de peso muestral (empiezan con WT).
    """
    df_clean = df.copy()
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    
    # Excluir columnas clave que legitimamente pueden tener estos valores
    exclude_prefixes = ['SEQN', 'WT', 'SDM']
    cols_to_check = [c for c in numeric_cols if not any(c.startswith(p) for p in exclude_prefixes)]
    
    total_replaced = 0
    for col in cols_to_check:
        mask = df_clean[col].isin(special_values)
        count = mask.sum()
        if count > 0:
            total_replaced += count
            df_clean.loc[mask, col] = np.nan
    
    print(f'Total de valores especiales recodificados a NaN: {total_replaced:,}')
    return df_clean

# IMPORTANTE: Los valores 7 y 9 solos son ambiguos en columnas con rangos amplios
# Por ello, solo aplicamos 7 y 9 a columnas que documentalmente usan esa codificacion.
# Para columnas de rango amplio (ej. peso, talla), solo recodificamos 77+ y 99+.
NHANES_SAFE_VALUES = [77, 99, 777, 999, 7777, 9999, 77777, 99999]

print('--- Recodificación DemoDietary ---')
df_dietary = recode_nhanes_special_values(df_dietary, NHANES_SAFE_VALUES)

print('\n--- Recodificación DemoExamination ---')
df_examination = recode_nhanes_special_values(df_examination, NHANES_SAFE_VALUES)

print('\n--- Recodificación DemoLaboratory ---')
df_laboratory = recode_nhanes_special_values(df_laboratory, NHANES_SAFE_VALUES)

--- Recodificación DemoDietary ---
Total de valores especiales recodificados a NaN: 3,879

--- Recodificación DemoExamination ---
Total de valores especiales recodificados a NaN: 6,738

--- Recodificación DemoLaboratory ---
Total de valores especiales recodificados a NaN: 3,555


## Paso 3.3: Eliminación de Columnas con Exceso de Nulidad

Eliminamos columnas donde el porcentaje de nulos supera un umbral configurable.
- **Umbral recomendado: 50%** (columnas con mas de la mitad de datos faltantes aportan poco valor predictivo).

In [7]:
NULL_THRESHOLD = 0.50  # 50%

def drop_high_null_columns(df: pd.DataFrame, threshold: float, name: str) -> pd.DataFrame:
    """Elimina columnas cuyo porcentaje de nulos supera el umbral."""
    null_pct = df.isnull().mean()
    cols_to_drop = null_pct[null_pct > threshold].index.tolist()
    
    # Nunca eliminar SEQN
    cols_to_drop = [c for c in cols_to_drop if c != 'SEQN']
    
    print(f'\n=== {name}: Eliminación de columnas con >{threshold*100:.0f}% nulos ===')
    print(f'Columnas antes: {len(df.columns)}')
    print(f'Columnas a eliminar: {len(cols_to_drop)}')
    
    df_clean = df.drop(columns=cols_to_drop)
    print(f'Columnas después: {len(df_clean.columns)}')
    
    return df_clean

df_dietary = drop_high_null_columns(df_dietary, NULL_THRESHOLD, 'DemoDietary')
df_examination = drop_high_null_columns(df_examination, NULL_THRESHOLD, 'DemoExamination')
df_laboratory = drop_high_null_columns(df_laboratory, NULL_THRESHOLD, 'DemoLaboratory')


=== DemoDietary: Eliminación de columnas con >50% nulos ===
Columnas antes: 718
Columnas a eliminar: 350
Columnas después: 368

=== DemoExamination: Eliminación de columnas con >50% nulos ===
Columnas antes: 1713
Columnas a eliminar: 1452
Columnas después: 261

=== DemoLaboratory: Eliminación de columnas con >50% nulos ===
Columnas antes: 698
Columnas a eliminar: 444
Columnas después: 254


## Paso 3.4: Eliminación de Filas con Exceso de Nulidad

Eliminamos filas donde el porcentaje de valores nulos sea demasiado alto.
- **Umbral recomendado para filas: 70%** (una persona con >70% de campos vacíos no aporta suficiente información).

In [8]:
ROW_NULL_THRESHOLD = 0.70  # 70%

def drop_high_null_rows(df: pd.DataFrame, threshold: float, name: str) -> pd.DataFrame:
    """Elimina filas cuyo porcentaje de nulos supera el umbral."""
    null_pct_per_row = df.isnull().mean(axis=1)
    rows_to_drop = null_pct_per_row[null_pct_per_row > threshold].index
    
    print(f'\n=== {name}: Eliminación de filas con >{threshold*100:.0f}% nulos ===')
    print(f'Filas antes: {len(df)}')
    print(f'Filas a eliminar: {len(rows_to_drop)}')
    
    df_clean = df.drop(index=rows_to_drop).reset_index(drop=True)
    print(f'Filas después: {len(df_clean)}')
    
    return df_clean

df_dietary = drop_high_null_rows(df_dietary, ROW_NULL_THRESHOLD, 'DemoDietary')
df_examination = drop_high_null_rows(df_examination, ROW_NULL_THRESHOLD, 'DemoExamination')
df_laboratory = drop_high_null_rows(df_laboratory, ROW_NULL_THRESHOLD, 'DemoLaboratory')


=== DemoDietary: Eliminación de filas con >70% nulos ===
Filas antes: 9254
Filas a eliminar: 1733
Filas después: 7521

=== DemoExamination: Eliminación de filas con >70% nulos ===
Filas antes: 9254
Filas a eliminar: 888
Filas después: 8366

=== DemoLaboratory: Eliminación de filas con >70% nulos ===
Filas antes: 9254
Filas a eliminar: 1737
Filas después: 7517


## Paso 3.5: Eliminación de Columnas Constantes o Casi-Constantes

Columnas con un solo valor único (o con >99% del mismo valor) no aportan poder predictivo.

In [9]:
def drop_low_variance_columns(df: pd.DataFrame, name: str, threshold: float = 0.99) -> pd.DataFrame:
    """Elimina columnas donde un solo valor domina mas del threshold."""
    cols_to_drop = []
    for col in df.columns:
        if col == 'SEQN':
            continue
        top_freq = df[col].value_counts(normalize=True, dropna=True)
        if len(top_freq) > 0 and top_freq.iloc[0] >= threshold:
            cols_to_drop.append(col)
    
    print(f'\n=== {name}: Columnas constantes o casi-constantes (>{threshold*100:.0f}% mismo valor) ===')
    print(f'Columnas a eliminar: {len(cols_to_drop)}')
    df_clean = df.drop(columns=cols_to_drop)
    print(f'Columnas restantes: {len(df_clean.columns)}')
    return df_clean

df_dietary = drop_low_variance_columns(df_dietary, 'DemoDietary')
df_examination = drop_low_variance_columns(df_examination, 'DemoExamination')
df_laboratory = drop_low_variance_columns(df_laboratory, 'DemoLaboratory')


=== DemoDietary: Columnas constantes o casi-constantes (>99% mismo valor) ===
Columnas a eliminar: 11
Columnas restantes: 357

=== DemoExamination: Columnas constantes o casi-constantes (>99% mismo valor) ===
Columnas a eliminar: 64
Columnas restantes: 197

=== DemoLaboratory: Columnas constantes o casi-constantes (>99% mismo valor) ===
Columnas a eliminar: 37
Columnas restantes: 217


## Paso 3.6: Reporte Final de Calidad

In [10]:
print('=' * 60)
print('REPORTE FINAL DE CALIDAD - FASE 3')
print('=' * 60)

for name, df in [('DemoDietary', df_dietary), ('DemoExamination', df_examination), ('DemoLaboratory', df_laboratory)]:
    total_cells = df.shape[0] * df.shape[1]
    total_nulls = df.isnull().sum().sum()
    null_pct_global = (total_nulls / total_cells * 100)
    
    print(f'\n--- {name} ---')
    print(f'Shape: {df.shape}')
    print(f'SEQN unicos: {df["SEQN"].nunique()}')
    print(f'Nulidad global: {null_pct_global:.2f}%')
    print(f'Columnas numericas: {len(df.select_dtypes(include=[np.number]).columns)}')
    print(f'Columnas categoricas: {len(df.select_dtypes(include=["object", "category"]).columns)}')

REPORTE FINAL DE CALIDAD - FASE 3

--- DemoDietary ---
Shape: (7521, 357)
SEQN unicos: 7521
Nulidad global: 6.92%
Columnas numericas: 357
Columnas categoricas: 0

--- DemoExamination ---
Shape: (8366, 197)
SEQN unicos: 8366
Nulidad global: 8.14%
Columnas numericas: 120
Columnas categoricas: 77

--- DemoLaboratory ---
Shape: (7517, 217)
SEQN unicos: 7517
Nulidad global: 12.60%
Columnas numericas: 217
Columnas categoricas: 0


## Paso 3.7: Guardado de Datasets Limpios en el Catálogo

Guardamos los resultados limpios usando el Data Catalog de Kedro.
Los nombres de salida serán:
- `demodietary_clean`
- `demoexamination_clean`
- `demolaboratory_clean`

In [11]:
# Guardado via Kedro Catalog
catalog.save('demodietary_clean', df_dietary)
print('Guardado: demodietary_clean')

catalog.save('demoexamination_clean', df_examination)
print('Guardado: demoexamination_clean')

catalog.save('demolaboratory_clean', df_laboratory)
print('Guardado: demolaboratory_clean')

print('\nFASE 3 COMPLETADA. Los 3 datasets limpios han sido persistidos en el catalogo.')

[06/13/26 18:13:44] INFO     Saving data to demodietary_clean (ParquetDataset)...              ]8;id=842776;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=137811;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1008\1008]8;;\

Guardado: demodietary_clean


[06/13/26 18:13:45] INFO     Saving data to demoexamination_clean (ParquetDataset)...          ]8;id=640744;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=845833;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1008\1008]8;;\

Guardado: demoexamination_clean


                    INFO     Saving data to demolaboratory_clean (ParquetDataset)...           ]8;id=989155;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=36795;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1008\1008]8;;\

Guardado: demolaboratory_clean

FASE 3 COMPLETADA. Los 3 datasets limpios han sido persistidos en el catalogo.
